In [1]:
import csv
import pandas as pd

In [2]:
file_path = "data/slurm_logs.csv"
clean_file_path = "data/prepped_logs.csv"

In [3]:
newline_replacement = " <NL> "
pipe_replacement = " <PIPE> "
valid_prefixes = ("default", "siteadmin")
max_record_len = 50000

In [4]:
def fix_record(record, expected_cols, command_idx):
    parts = record.split("|")

    if len(parts) == expected_cols:
        return record, "ok"

    if len(parts) < expected_cols:
        return record, "short"

    tail_len = expected_cols - command_idx - 1

    head = parts[:command_idx]
    tail = parts[-tail_len:] if tail_len > 0 else []
    cmd_parts = parts[command_idx:len(parts) - tail_len] if tail_len > 0 else parts[command_idx:]

    repaired_command = pipe_replacement.join(cmd_parts)
    fixed_parts = head + [repaired_command] + tail

    if len(fixed_parts) != expected_cols:
        return record, "bad"

    return "|".join(fixed_parts), "fixed"

In [5]:
with open(file_path, "r", encoding="latin1", errors="replace") as infile:
    raw_header = infile.readline().rstrip("\r\n").replace("\x00", "")
    header_cols = raw_header.split("|")
    expected_cols = len(header_cols)

    command_candidates = ["Command", "SubmitLine", "submitline", "command", "Cmd"]
    command_idx = None
    for name in command_candidates:
        if name in header_cols:
            command_idx = header_cols.index(name)
            break

    if command_idx is None:
        command_idx = 89
        print(f"Command column not found in header; using fallback index {command_idx}")

    current_record = None
    records_written = 1   # header
    continuation_lines = 0
    skipped_long_records = 0
    fixed_pipe_records = 0
    bad_records = 0

    with open(clean_file_path, "w", encoding="utf-8", newline="") as outfile:
        outfile.write(raw_header + "\n")

        for raw_line in infile:
            line = raw_line.rstrip("\r\n").replace("\x00", "")

            if current_record is None and line == "":
                continue

            if line.startswith(valid_prefixes):
                if current_record is not None:
                    if len(current_record) >= max_record_len:
                        skipped_long_records += 1
                    else:
                        fixed, status = fix_record(current_record, expected_cols, command_idx)
                        if status == "ok":
                            outfile.write(fixed + "\n")
                            records_written += 1
                        elif status == "fixed":
                            outfile.write(fixed + "\n")
                            records_written += 1
                            fixed_pipe_records += 1
                        else:
                            bad_records += 1

                current_record = line
            else:
                if current_record is None:
                    continue
                current_record += newline_replacement + line
                continuation_lines += 1

        if current_record is not None:
            if len(current_record) >= max_record_len:
                skipped_long_records += 1
            else:
                fixed, status = fix_record(current_record, expected_cols, command_idx)
                if status == "ok":
                    outfile.write(fixed + "\n")
                    records_written += 1
                elif status == "fixed":
                    outfile.write(fixed + "\n")
                    records_written += 1
                    fixed_pipe_records += 1
                else:
                    bad_records += 1

print(f"Expected columns: {expected_cols}")
print(f"Done. Wrote {records_written} lines including header.")
print(f"Folded {continuation_lines} continuation lines into previous records.")
print(f"Fixed shell-pipe records: {fixed_pipe_records}")
print(f"Skipped oversized records: {skipped_long_records}")
print(f"Unresolved bad records: {bad_records}")

Expected columns: 121
Done. Wrote 10888403 lines including header.
Folded 94497 continuation lines into previous records.
Fixed shell-pipe records: 76
Skipped oversized records: 0
Unresolved bad records: 0


In [6]:
expected_cols = None
bad_lines = []

with open(clean_file_path, "r", encoding="utf-8", errors="replace") as f:
    for lineno, line in enumerate(f, start=1):
        cols = line.rstrip("\n").split("|")
        ncols = len(cols)

        if lineno == 1:
            expected_cols = ncols
            print(f"Header line has {expected_cols} columns")
            continue

        if ncols != expected_cols:
            bad_lines.append((lineno, ncols))

print(f"Expected columns: {expected_cols}")
print(f"Bad lines found: {len(bad_lines)}")

if bad_lines:
    print("First few bad lines:")
    for lineno, ncols in bad_lines[:20]:
        print(f"Line {lineno}: {ncols} columns")
else:
    print("All lines have the same number of columns.")

Header line has 121 columns
Expected columns: 121
Bad lines found: 0
All lines have the same number of columns.


In [7]:
df = pd.read_csv(
    clean_file_path,
    sep="|",
    engine="c",
    low_memory=False
)
df.to_parquet("data/slurm.parquet", engine='pyarrow', index=False)